In [ ]:
import numpy as np
import jax.numpy as jnp
import dynamiqs as dq
import matplotlib.pyplot as plt

dq.set_precision("double")

g2 = 2 * np.pi * 0.763
kappa_b = 2 * np.pi * 2.6
kappa_a = 2 * np.pi * 9.3e-3
alpha = 2.1
eps_d = alpha**2 * g2
eps_Z = 0.1 * g2

t = np.linspace(0.0, 3.0, 601)
method = dq.method.Tsit5(rtol=1e-8, atol=1e-10)


def buffer_field(eps, sign, Na=24, Nb=6):
    a, b = dq.destroy(Na, Nb)
    H = g2 * (a @ a @ b.dag()) + g2 * (a.dag() @ a.dag() @ b) - eps_d * (b + b.dag()) + eps * (a + a.dag())
    jump_ops = [jnp.sqrt(kappa_b) * b, jnp.sqrt(kappa_a) * a]
    psi0 = dq.tensor(dq.coherent(Na, sign * alpha), dq.fock(Nb, 0))
    result = dq.mesolve(H, jump_ops, psi0, t, exp_ops=[b], method=method, progress_meter=False)
    return np.asarray(result.expects[0])


runs = {(eps, sign): buffer_field(eps, sign) for eps in (0.0, eps_Z) for sign in (+1, -1)}

for (eps, sign), b in runs.items():
    start = "+alpha" if sign > 0 else "-alpha"
    print(f"eps_Z = {eps / g2:.1f} g2, start {start}: <b>_ss = {b[-1].real:+.5f} {b[-1].imag:+.5f}i")

b_lin = eps_Z / (2 * alpha * g2)
b_plus = runs[(eps_Z, +1)].real
b_minus = runs[(eps_Z, -1)].real
print(f"Linear estimate: Re<b>_ss = -/+{b_lin:.5f}, ratio simulated/estimate = {b_plus[-1] / -b_lin:.4f}")
print(f"Readout contrast Re<b>(-alpha) - Re<b>(+alpha) = {b_minus[-1] - b_plus[-1]:.5f}")

ss = b_plus[-1]
overshoot = np.max(np.abs(b_plus)) / abs(ss) - 1
t_peak = t[np.argmax(np.abs(b_plus))]
t_settle = t[np.nonzero(np.abs(b_plus - ss) > 0.01 * abs(ss))[0][-1] + 1]
print(f"First overshoot {100 * overshoot:.0f}% at t = {t_peak:.3f} us; within 1% of final value after t = {t_settle:.2f} us")

for eps in (0.0, eps_Z):
    change = np.max(np.abs(buffer_field(eps, +1, Na=32, Nb=8) - runs[(eps, +1)]))
    print(f"Convergence, eps_Z = {eps / g2:.1f} g2: max |<b>(Na=32,Nb=8) - <b>(Na=24,Nb=6)| = {change:.1e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
titles = [r"Readout drive off ($\epsilon_Z = 0$)", r"Readout drive on ($\epsilon_Z = 0.1\,g_2$)"]
for ax, eps, title in zip(axes, (0.0, eps_Z), titles):
    ax.plot(t, runs[(eps, +1)].real, color="#2a78d6", lw=2, label=r"start in $|{+}\alpha\rangle$")
    ax.plot(t, runs[(eps, -1)].real, color="#eb6834", lw=2, ls="--", label=r"start in $|{-}\alpha\rangle$")
    ax.set_title(title)
    ax.set_xlabel(r"$t$ ($\mu$s)")
    ax.set_xlim(0, 3)
    ax.grid(alpha=0.3)
axes[0].set_ylabel(r"Re$\langle b\rangle$")
axes[0].text(0.5, 0.88, "no contrast: both starts give the same buffer field",
             transform=axes[0].transAxes, ha="center", fontsize=9)
axes[1].axhline(-b_lin, color="k", ls=":", lw=1, label=r"linear estimate $\mp\epsilon_Z/(2\alpha g_2)$")
axes[1].axhline(b_lin, color="k", ls=":", lw=1)
axes[1].axvline(t_settle, color="0.4", ls="-.", lw=1, label=rf"within 1% after {t_settle:.2f} $\mu$s")
axes[1].text(2.95, -0.012, f"contrast = {b_minus[-1] - b_plus[-1]:.4f}", ha="right", va="center", fontsize=9)
fig.legend(*axes[1].get_legend_handles_labels(), loc="lower center", ncol=4, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig("task23_readout.png", dpi=160)
print("Saved task23_readout.png")
